Checkpoint creation

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbw_taxrisk_dev.bronze.checkpoints
COMMENT 'Auto Loader checkpoints and inferred schemas';

Ingestion function

In [0]:
from pyspark.sql import functions as F

CATALOG = "dbw_taxrisk_dev"
LANDING = f"/Volumes/{CATALOG}/bronze/landing"
CHK     = f"/Volumes/{CATALOG}/bronze/checkpoints"

def ingest(source: str, target: str, is_synthetic: bool = True):
    """Incrementally load LANDING/<source>/ into bronze.<target> with Auto Loader."""
    stream = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{CHK}/{target}/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(f"{LANDING}/{source}/")
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("is_synthetic", F.lit(is_synthetic))
    )
    q = (
        stream.writeStream
        .option("checkpointLocation", f"{CHK}/{target}/checkpoint")
        .trigger(availableNow=True)      # process what is there, then stop
        .toTable(f"{CATALOG}.bronze.{target}")
    )
    q.awaitTermination()
    n = spark.table(f"{CATALOG}.bronze.{target}").count()
    print(f"bronze.{target}: {n} rows")

ingest("posts",    "posts")
ingest("venues",   "venues")
ingest("registry", "registry")

Verify, including the idempotency test

In [0]:
# 1. Re-run: counts must NOT change
ingest("posts", "posts")

# 2. Inspect schema and a sample
spark.table(f"{CATALOG}.bronze.posts").printSchema()
display(spark.table(f"{CATALOG}.bronze.posts").limit(3))

# 3. Ground truth must not be in Bronze
display(spark.sql(f"SHOW TABLES IN {CATALOG}.bronze"))